In [ ]:
import QuantLib as ql
import plotly.express as px

In [ ]:
import numpy as np
import polars as pl

In [ ]:
layout_dict = dict(
    margin=dict(l=20, r=20, t=40, b=20),
    width=600,
    height=400,
    paper_bgcolor="LightSteelBlue",
    title_font_size=14,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
)

In [ ]:
calc_date = ql.Date(30, 6, 2019)
# ql.Settings.instance().evaluationDate = calc_date
# bond creation

settlement_days = 2
face_amount = 100

issue_date = ql.Date(30, 6, 2019)
maturity_date = ql.Date(15, 9, 2023)
tenor = ql.Period(ql.Semiannual)
calendar = ql.UnitedStates(ql.UnitedStates.Settlement)
accrual_convention = ql.Unadjusted
Rule = ql.DateGeneration.Backward
endofMonth = False
schedule = ql.Schedule(
    issue_date,
    maturity_date,
    tenor,
    calendar,
    accrual_convention,
    accrual_convention,
    Rule,
    endofMonth,
)


coupon = 0.07
day_count = ql.Actual360()  # ql.ActualActual(ql.ActualActual.Bond)

callability_schedule = ql.CallabilitySchedule()
null_calendar = ql.NullCalendar()
for strike, dt in zip(
    [106., 104., 102., 100., 100.0],
    [ql.Date(15, 9, 2019), ql.Date(15, 9, 2020), ql.Date(15, 9, 2021), ql.Date(15, 9, 2022), ql.Date(15, 9, 2023)],
):
    callability_price = ql.BondPrice(strike, ql.BondPrice.Clean)
    callability_schedule.append(
        ql.Callability(callability_price, ql.Callability.Call, dt)
    )


bond = ql.CallableFixedRateBond(
    settlement_days,
    face_amount,
    schedule,
    [coupon],
    day_count,
    ql.Following,
    face_amount,  # redemption
    calc_date,
    callability_schedule,
)

In [ ]:
calc_date = ql.Date(30, 6, 2019)
ql.Settings.instance().evaluationDate = calc_date

# bond valuation
# HullWhite dX = a*(theta - r) + s*dW, theta is term structure..
a = 0.2
sigma = 0.05
grid_points = 200
flat_rate = 0.02

curve = ql.FlatForward(
    settlement_days, ql.TARGET(), ql.QuoteHandle(ql.SimpleQuote(flat_rate)), day_count
)
ts_handle = ql.YieldTermStructureHandle(curve)


def value_bond(a, s, ts_handle, grid_points, bond):
    model = ql.HullWhite(ts_handle, a, s)
    engine = ql.TreeCallableFixedRateBondEngine(model, grid_points)
    bond.setPricingEngine(engine)
    return bond


compounding = ql.Compounded
frequency = ql.Annual

bondprice = value_bond(a, sigma, ts_handle, grid_points, bond)
px_at_250bps = bond.cleanPriceOAS(
    0.0250, ts_handle, day_count, compounding, frequency
)
OAS = bondprice.OAS(100., ts_handle, day_count, compounding, frequency)

print(px_at_250bps)
print(OAS)


# bond_yield = bondprice.bondYield(
#     ql.BondPrice(100, ql.BondPrice.Clean), day_count, compounding, frequency
# )
# print(bond_yield)

In [ ]:
# step 3: check PDE numerically

In [ ]:
# zspread
oas = np.linspace(-0.03+1e-4, 0.21, 600)
prices = np.empty(600, float)
for i, o in enumerate(oas):
    prices[i] = bond.cleanPriceOAS(o, ts_handle, day_count, compounding, frequency)

In [ ]:
fig = px.line(
    pl.DataFrame(dict(oas = oas, price = prices)), 'oas', 'price'
)

fig.update_layout(**layout_dict)

In [ ]:
# take the point oas = 60bps -- > price at constant OAS. Produce daily TS for Z-spreads, and prices, and OASs

In [ ]:
bf = ql.BondFunctions
qd = ql.DateParser.parseFormatted

# Conventions
accrual_convention = ql.Unadjusted
Rule = ql.DateGeneration.Backward
endofMonth = False
firstDate = None

# OAS
compounding = ql.Compounded
frequency = ql.Annual

calendar = ql.UnitedStates(ql.UnitedStates.Settlement)

a = 0.1
sigma = 0.1
grid_points = 100
face_amount = 100
mkt_price = 78

contract = {
    "IssueDate": ql.Date(30, 6, 2016),
    "MaturityDate": ql.Date(15, 6, 2023),
    "SettlementDays": 2,
    "FirstCouponDate": ql.Date(15, 12, 2016),
    "NextToLastCouponDate": ql.Date(15, 12, 2022),
    "RealValue": 0.0675,
    "FirstCallDate": ql.Date(15, 6, 2019),
    "OptionalityEndDate": ql.Date(15, 6, 2023),
    "OperatingCountry": "US",
    "StrikeDate": [ql.Date(15, 6, 2019), ql.Date(15, 6, 2020), ql.Date(15, 6, 2021)],
    "OptionalityType": ["Call", "Call", "Call"],
    "NoticeDays": [30, 30, 30],
    "StrikePrice": [103.375, 101.688, 100.0],
}

# Here is the zero curve:

times = np.array(
    [
        "0 MO",
        "1 MO",
        "2 MO",
        "3 MO",
        "6 MO",
        "1 YR",
        "2 YR",
        "3 YR",
        "5 YR",
        "7 YR",
        "10 YR",
        "20 YR",
        "30 YR",
    ],
    dtype=object,
)

dates = [
    ql.Date(9, 2, 2017),
    ql.Date(9, 3, 2017),
    ql.Date(9, 4, 2017),
    ql.Date(9, 5, 2017),
    ql.Date(9, 8, 2017),
    ql.Date(9, 2, 2018),
    ql.Date(9, 2, 2019),
    ql.Date(9, 2, 2020),
    ql.Date(9, 2, 2022),
    ql.Date(9, 2, 2024),
    ql.Date(9, 2, 2027),
    ql.Date(9, 2, 2037),
    ql.Date(9, 2, 2047),
]

rates = np.array(
    [0.51, 0.51, 0.525, 0.54, 0.64, 0.8, 1.2, 1.46, 1.88, 2.2, 2.4, 2.74, 3.02]
)

day_count = ql.ActualActual(ql.ActualActual.Bond)
calc_date = ql.Date(9, 2, 2017)
ql.Settings.instance().evaluationDate = calc_date

issue_date = contract["IssueDate"]
maturity_date = contract["MaturityDate"]
tenor = ql.Period(ql.Semiannual)

coupon = contract["RealValue"]
settlement_days = contract["SettlementDays"]

# Determine Schedule
schedule = ql.Schedule(
    issue_date,
    maturity_date,
    tenor,
    calendar,
    accrual_convention,
    accrual_convention,
    Rule,
    endofMonth,
)

# Initiate Zero Curve
curve = ql.ZeroCurve(
    dates, rates, ql.ActualActual(ql.ActualActual.Bond), calendar, ql.Linear()
)
curve.enableExtrapolation()
ts_handle = ql.YieldTermStructureHandle(curve)


def get_call_schedule(df, period=ql.Period(ql.Annual)):
    dates = df["StrikeDate"]
    prices = df["StrikePrice"]
    callability_schedule = ql.CallabilitySchedule()
    null_calendar = ql.NullCalendar()
    call_date = df["StrikeDate"][0]
    for time in range(len(df["StrikeDate"])):
        callability_price = ql.BondPrice(prices[time], ql.BondPrice.Clean)
        callability_schedule.append(
            ql.Callability(callability_price, ql.Callability.Call, dates[time])
        )
        call_date = null_calendar.advance(call_date, period)
    return callability_schedule


callability_schedule = get_call_schedule(contract)

bond = ql.CallableFixedRateBond(
    settlement_days,
    face_amount,
    schedule,
    [coupon],
    day_count,
    ql.Following,
    face_amount,
    calc_date,
    callability_schedule,
)


def value_bond(a, s, ts_handle, grid_points, bond):
    model = ql.HullWhite(ts_handle, a, s)
    engine = ql.TreeCallableFixedRateBondEngine(model, grid_points)
    bond.setPricingEngine(engine)
    return bond


bondprice = value_bond(a, sigma, ts_handle, grid_points, bond)
OAS = bondprice.OAS(mkt_price, ts_handle, day_count, compounding, frequency)
bond_yield = bondprice.bondYield(day_count, compounding, frequency)

print(OAS)
print(bond_yield)